# Module 1: Telco Customer Churn Prediction & Decision Modeling

Welcome to **Module 1**! In this practical lab, you will analyze customer churn for a telecommunications provider (`Telco-Customer-Churn.csv` — 7,043 customers, 21 columns).

### Business Brief:
A telecom company loses ~25% of its customers annually. Winning new customers costs 5x more than retaining existing ones. Your brief:
> *"Predict which customers are likely to churn, identify root causes, and determine the optimal intervention cutoff for retention calls."*

---

### Assignment Tasks Overview:
1. **Task 1**: Load `Telco-Customer-Churn.csv`, inspect schema, convert `TotalCharges` strings to float, and impute missing values.
2. **Task 2**: Exploratory Data Analysis — Analyze churn rates across contract types, tenure buckets, and monthly charges.
3. **Task 3**: Preprocessing & Feature Encoding — One-hot encode categorical features and scale numeric variables.
4. **Task 4**: Machine Learning Model Training — Train Logistic Regression and Random Forest classifiers.
5. **Task 5**: Business Decision Threshold Modeling — Evaluate Precision-Recall trade-offs to maximize retention campaign ROI.

In [2]:
# Task 1: Load Telco Churn Dataset & Handle Data Types
import pandas as pd
import numpy as np

# Load dataset (pre-mounted in workspace)
df = pd.read_csv('Telco-Customer-Churn.csv')

# Clean TotalCharges (spaces/blank strings -> NaN -> float)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Encode target variable
df['Churn_Numeric'] = (df['Churn'] == 'Yes').astype(int)

print(f"Dataset Shape: {df.shape}")
print("Churn Distribution:\n", df['Churn'].value_counts(normalize=True))
df.head()

Dataset Shape: (7043, 22)
Churn Distribution:
 No     0.73463
Yes    0.26537
Name: Churn, dtype: float64



<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>customerID</th>
      <th>gender</th>
      <th>SeniorCitizen</th>
      <th>Partner</th>
      <th>Dependents</th>
      <th>tenure</th>
      <th>PhoneService</th>
      <th>MultipleLines</th>
      <th>InternetService</th>
      <th>OnlineSecurity</th>
      <th>OnlineBackup</th>
      <th>DeviceProtection</th>
      <th>TechSupport</th>
      <th>StreamingTV</th>
      <th>StreamingMovies</th>
      <th>Contract</th>
      <th>PaperlessBilling</th>
      <th>PaymentMethod</th>
      <th>MonthlyCharges</th>
      <th>TotalCharges</th>
      <th>Churn</th>
      <th>Churn_Numeric</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
     

### Task 2: Exploratory Data Analysis & Churn Drivers
Analyze how Contract Type, Tenure, and Internet Service impact churn rates.

In [3]:
# Task 2: Churn Analysis by Contract & Service
contract_churn = df.groupby('Contract')['Churn_Numeric'].agg(['count', 'mean']).rename(columns={'mean': 'Churn_Rate'})
contract_churn['Churn_Rate'] = (contract_churn['Churn_Rate'] * 100).round(2).astype(str) + '%'

print("--- Churn Rate by Contract Type ---")
print(contract_churn)

# Tenure Grouping
df['TenureGroup'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72], labels=['0-1 yr', '1-2 yrs', '2-4 yrs', '4+ yrs'])
tenure_churn = df.groupby('TenureGroup')['Churn_Numeric'].mean() * 100
print("\n--- Churn Rate by Tenure Group ---")
print(tenure_churn.round(2).astype(str) + '%')

--- Churn Rate by Contract Type ---
                count Churn_Rate
Contract                        
Month-to-month   3875     42.71%
One year         1473     11.27%
Two year         1695      2.83%

--- Churn Rate by Tenure Group ---
TenureGroup
0-1 yr     47.68%
1-2 yrs    28.71%
2-4 yrs    20.39%
4+ yrs      9.51%
Name: Churn_Numeric, dtype: object



### Task 3: Model Training & Evaluation
Train a Random Forest classifier and generate a Confusion Matrix.

In [4]:
# Task 3: Preprocessing & Baseline Classifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

feature_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
X = df[feature_cols]
y = df['Churn_Numeric']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.87      0.84      1035
           1       0.55      0.44      0.48       374

    accuracy                           0.75      1409
   macro avg       0.68      0.65      0.66      1409
weighted avg       0.74      0.75      0.74      1409




In [5]:
# Task 4: Model Training & Comparison

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# -----------------------------
# Logistic Regression
# -----------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(random_state=42, max_iter=1000)
log_model.fit(X_train_scaled, y_train)

log_pred = log_model.predict(X_test_scaled)

print("=== Logistic Regression ===")
print(classification_report(y_test, log_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, log_pred))
print("Accuracy:", accuracy_score(y_test, log_pred))


# -----------------------------
# Random Forest
# -----------------------------
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("\n=== Random Forest ===")
print(classification_report(y_test, rf_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_pred))
print("Accuracy:", accuracy_score(y_test, rf_pred))

=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.81      0.90      0.85      1035
           1       0.60      0.42      0.49       374

    accuracy                           0.77      1409
   macro avg       0.71      0.66      0.67      1409
weighted avg       0.76      0.77      0.76      1409

Confusion Matrix:
[[931 104]
 [217 157]]
Accuracy: 0.7721788502484032

=== Random Forest ===
              precision    recall  f1-score   support

           0       0.81      0.87      0.84      1035
           1       0.55      0.44      0.48       374

    accuracy                           0.75      1409
   macro avg       0.68      0.65      0.66      1409
weighted avg       0.74      0.75      0.74      1409

Confusion Matrix:
[[899 136]
 [211 163]]
Accuracy: 0.7537260468417317



In [6]:
# Task 5: Business Decision Threshold Modeling

import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score

# Business assumptions
retention_value = 5
campaign_cost = 1

# Predicted probabilities
log_prob = log_model.predict_proba(X_test_scaled)[:, 1]
rf_prob = rf_model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.10, 0.91, 0.05)


def evaluate_threshold(y_true, probabilities, model_name):
    results = []

    for threshold in thresholds:

        predicted_churn = (probabilities >= threshold).astype(int)

        precision = precision_score(
            y_true,
            predicted_churn,
            zero_division=0
        )

        recall = recall_score(
            y_true,
            predicted_churn,
            zero_division=0
        )

        contacted = predicted_churn.sum()
        true_churners = ((predicted_churn == 1) & (y_true == 1)).sum()

        net_value = (
            true_churners * retention_value
            - contacted * campaign_cost
        )

        results.append({
            'Threshold': round(threshold, 2),
            'Precision': round(precision, 3),
            'Recall': round(recall, 3),
            'Customers Contacted': contacted,
            'True Churners': true_churners,
            'Net Value': net_value
        })

    results_df = pd.DataFrame(results)

    best_result = results_df.loc[
        results_df['Net Value'].idxmax()
    ]

    print(f"\n=== {model_name} ===")
    print(results_df.to_string(index=False))

    print("\nBest Threshold:")
    print(best_result)

    return results_df


# Logistic Regression
log_results = evaluate_threshold(
    y_test,
    log_prob,
    "Logistic Regression"
)


# Random Forest
rf_results = evaluate_threshold(
    y_test,
    rf_prob,
    "Random Forest"
)


=== Logistic Regression ===
 Threshold  Precision  Recall  Customers Contacted  True Churners  Net Value
      0.10      0.356   0.957                 1006            358        784
      0.15      0.403   0.914                  848            342        862
      0.20      0.445   0.872                  733            326        897
      0.25      0.473   0.757                  598            283        817
      0.30      0.493   0.693                  525            259        770
      0.35      0.514   0.639                  465            239        730
      0.40      0.554   0.599                  404            224        716
      0.45      0.583   0.508                  326            190        624
      0.50      0.602   0.420                  261            157        524
      0.55      0.664   0.385                  217            144        503
      0.60      0.723   0.307                  159            115        416
      0.65      0.670   0.163                  